# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of a specific content item for a specific search query over a 90-day reporting window.

Time window: The reporting period is 90 days, shown by the window_start and window_end columns. In this dataset, the dates are 2026-04-02 to 2026-06-30.

content_hash_id identifies the content (article/page).
query_hash_id identifies the search query.
window_start = 2026-04-02
window_end   =   2026-06-30
Performance metrics such as impressions_90d, clicks_90d, and avg_position_90d are measured within that 90-day period.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES :
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
content_age_days
days_since_last_update
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
age_tier
freshness_tier
word_count_tier
char_count_tier
impression_tier
position_tier

LABEL :
is_declining_label

CONTEXT :
content_id
client_id

EXCLUDED :
Data leakage
trend_direction
trend_pct
impressions_last_30d
clicks_last_30d
sessions_last_30d

Metadata :
provider_used
model_used

   

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# Basic checks for the starter dataset

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nUnique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nContent types:")
print(df["content_type"].value_counts())

print("\nMissing values in key keyword fields:")
print(df[[
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "main_intent"
]].isna().sum())

Rows: 30000
Columns: 44

Unique content IDs: 30000
Unique clients: 32

Content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Missing values in key keyword fields:
search_volume        2468
competition          2468
competition_level    2610
cpc                  2468
main_intent          2374
dtype: int64


Verification: The dataset contains 30,000 rows and 44 columns, with 30,000 unique content items across 32 clients. The dataset contains three content types: keyword articles, Feedly articles, and comparison articles. Missing values are present in the keyword-context fields, confirming that these fields are not available for every content type.

In [12]:
# Verify the label and main 90-day performance fields

print("90-day performance columns:")
print([
    col for col in df.columns
    if col.endswith("_90d")
])

print("\nTrend direction values:")
print(df["trend_direction"].value_counts())

print("\nDeclining pages:")
print((df["trend_direction"] == "down").sum())

print("\nLabel percentage:")
print((df["trend_direction"] == "down").mean() * 100)

print("\nMissing values in main performance fields:")
print(df[
    [
        "impressions_90d",
        "clicks_90d",
        "pageviews_90d",
        "sessions_90d",
        "users_90d",
        "engaged_sessions_90d",
        "ai_sessions_90d"
    ]
].isna().sum())

90-day performance columns:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining pages:
16262

Label percentage:
54.20666666666667

Missing values in main performance fields:
impressions_90d         0
clicks_90d              0
pageviews_90d           0
sessions_90d            0
users_90d               0
engaged_sessions_90d    0
ai_sessions_90d         0
dtype: int64


Verification: The dataset contains all required 90-day performance fields with no missing values. The trend_direction label is present for all 30,000 records, with 16,262 pages (54.21%) classified as declining (down). This confirms that the target label is populated and suitable for the next stage of analysis.

In [13]:
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

Duplicate content IDs: 0


Verification: No duplicate content_id values were found. This confirms that each content item appears only once in the starter dataset, consistent with the stated one-row-per-content-item grain.

In [14]:
print("Content IDs:", df["content_id"].nunique())
print("Client IDs:", df["client_id"].nunique())

print("\nContent items per client:")
print(df.groupby("client_id")["content_id"].nunique().describe())

Content IDs: 30000
Client IDs: 32

Content items per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: content_id, dtype: float64


Verification: The dataset contains 30,000 unique content items across 32 clients. Each client has at least one associated content item, confirming that client_id is populated and that content is distributed across the client dimension. The distribution is uneven, ranging from 3 to 7,008 content items per client.

In [15]:
print("Days since last update:")
print(df["days_since_last_update"].describe())

print("\nMissing values:")
print(df["days_since_last_update"].isna().sum())

print("\nMinimum:", df["days_since_last_update"].min())
print("Maximum:", df["days_since_last_update"].max())

Days since last update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Missing values:
0

Minimum: 1
Maximum: 373


Verification: days_since_last_update is complete for all 30,000 records, with no missing values. Values range from 1 to 373 days, confirming that the dataset contains a valid update-recency field for assessing content freshness.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [16]:
# Section 4: Data limits

print("=== DATA LIMITS CHECK ===")

# 1. Client balance
print("\n1. Client balance:")
print(df["client_id"].value_counts().describe())

# 2. Identify source-related columns
source_cols = [c for c in df.columns if "source" in c.lower() or "gsc" in c.lower()]
print("\n2. Possible source/GSC columns:")
print(source_cols)

# 3. Identify date/window-related columns
date_cols = [
    c for c in df.columns
    if any(word in c.lower() for word in ["date", "window", "update", "start", "end"])
]
print("\n3. Date/window-related columns:")
print(date_cols)

# 4. Update-history distribution
print("\n4. Update history:")
print(df["days_since_last_update"].describe())

=== DATA LIMITS CHECK ===

1. Client balance:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64

2. Possible source/GSC columns:
[]

3. Date/window-related columns:
['days_since_last_update', 'trend_direction', 'trend_pct']

4. Update history:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64


### Data limitations

This dataset supports analysis of content performance and update recency, but it has important limitations.

- **Client balance:** The dataset contains 32 clients, but the number of content items per client varies substantially (3 to 7,008). Results may therefore be influenced by clients with much larger numbers of records.
- **Source/GSC coverage:** No explicit source or Google Search Console (GSC) columns were identified in the available dataset, so GSC-specific coverage and history cannot be directly verified from these records.
- **Update history:** The `days_since_last_update` field is complete, but values range from 1 to 373 days. This provides information about recency, not a complete historical timeline of every content update.
- **Window overlap:** The available fields do not provide enough information to independently verify whether performance windows overlap across records.

These limitations mean that the dataset is suitable for descriptive analysis and decision-support, but conclusions should not be treated as proof of causation or as a complete historical record.

Verification: The dataset is unbalanced across clients, with the number of content items per client ranging from 3 to 7,008. This means client-level comparisons may be influenced by unequal representation. The dataset also contains `days_since_last_update` for all 30,000 records, ranging from 1 to 373 days, but the available columns do not directly identify source history such as GSC coverage or explicit window boundaries. Therefore, conclusions about those aspects should be treated as dataset-design limitations rather than directly measured fields.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.